# Threads and the interpreter lock

Section 14.1 of the notes.

Threads share the memory of one program, but only the thread holding the
interpreter lock may execute Python instructions. So threads do not make Python
arithmetic faster. They help when the program waits, and when the work happens
inside compiled code that releases the lock.

In [ ]:
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np

import tasks

print("cores:", os.cpu_count())
print("python:", sys.version.split()[0])
print("lock enabled:", getattr(sys, "_is_gil_enabled", lambda: True)())


def wall(fn):
    t0 = time.perf_counter()
    fn()
    return time.perf_counter() - t0

## 1. Python arithmetic gains nothing

In [ ]:
N = 8

t_seq = wall(lambda: [tasks.cpu_work() for _ in range(N)])


def with_threads(k):
    with ThreadPoolExecutor(max_workers=k) as pool:
        list(pool.map(tasks.cpu_work, [tasks.CPU_SIZE] * N))


t_thr = wall(lambda: with_threads(N))
print(f"sequential {t_seq:5.2f} s")
print(f"{N} threads  {t_thr:5.2f} s   speed-up {t_seq / t_thr:.2f}x")

## 2. Waiting gains a lot

`tasks.wait` stands for reading a file or waiting for an answer. A waiting
thread releases the lock, so the waits overlap.

In [ ]:
t_seq = wall(lambda: [tasks.wait(0.2) for _ in range(N)])


def wait_threads(k):
    with ThreadPoolExecutor(max_workers=k) as pool:
        list(pool.map(tasks.wait, [0.2] * N))


t_thr = wall(lambda: wait_threads(N))
print(f"sequential {t_seq:5.2f} s")
print(f"{N} threads  {t_thr:5.2f} s   speed-up {t_seq / t_thr:.2f}x")

## 3. Work that leaves Python gains too

The compiled loop behind `np.sin` releases the lock while it runs, so several
threads can execute it at the same time. The gain stops well below the number
of threads, because the arrays no longer fit in the cache and the work becomes
limited by the memory rather than by the processor.

In [ ]:
arrays = [np.random.default_rng(k).standard_normal(20_000_000) for k in range(N)]

t_seq = wall(lambda: [tasks.elementwise(a) for a in arrays])


def elementwise_threads(k):
    with ThreadPoolExecutor(max_workers=k) as pool:
        list(pool.map(tasks.elementwise, arrays))


t_thr = wall(lambda: elementwise_threads(N))
print(f"sequential {t_seq:5.2f} s")
print(f"{N} threads  {t_thr:5.2f} s   speed-up {t_seq / t_thr:.2f}x")

## 4. Getting results as they finish

`map` returns results in the order of the arguments. `submit` with
`as_completed` returns them in the order in which they finish, which allows
progress to be reported while the rest are still running.

In [ ]:
delays = [0.4, 0.1, 0.3, 0.2]

with ThreadPoolExecutor(max_workers=4) as pool:
    print("map:        ", list(pool.map(tasks.wait, delays)))

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(tasks.wait, d) for d in delays]
    print("as_completed:", [f.result() for f in as_completed(futures)])